# Step 4: Deploy MCP Server

Deploy the MCP Server to AgentCore Runtime.

## Prerequisites

- ✅ Run `02-deploy-cognito.ipynb` first
- ✅ Docker installed and running

## What This Notebook Does

1. Deploys MCP Server to AgentCore Runtime
2. Configures Lake Formation RLS integration
3. Saves Runtime ARN to SSM

## Next Notebook

- **04-deploy-gateway.ipynb**

In [1]:
import sys
sys.path.insert(0, '.')  # Add current directory to path
from pathlib import Path
from aws_session_utils import get_aws_session

# Get validated AWS session with SSO support
session, region, account_id = get_aws_session()

# Initialize AWS clients
ssm_client = session.client('ssm', region_name=region)

# Load RLS Role ARN from SSM
try:
    rls_role_arn = ssm_client.get_parameter(
        Name='/app/lakehouse-agent/rls-role-arn'
    )['Parameter']['Value']
    print('✅ Setup complete')
    print(f'   Region: {region}')
    print(f'   RLS Role: {rls_role_arn}')
except ssm_client.exceptions.ParameterNotFound:
    print('⚠️  RLS Role ARN not found in SSM')
    print('   Will be configured during Lake Formation setup')
    rls_role_arn = None

🔍 Using default AWS credentials (no profile specified)
⚠️  No AWS region configured, using default: us-east-1
   To set your region:
   - Environment variable: export AWS_DEFAULT_REGION=your-region
   - AWS CLI: aws configure set region your-region

✅ AWS Credentials Validated
   Region: us-east-1
   Account ID: XXXXXXXXXXXX
   Profile: default
   Auth method: AWS SSO

✅ Setup complete
   Region: us-east-1
   RLS Role: arn:aws:iam::ACCOUNT_ID:role/lakehouse-rls-role


## Step 1: Deploy MCP Server

**Note**: This requires Docker to be running.

In [7]:
import subprocess

# Run deploy_runtime.py with --yes flag to skip interactive confirmation
# This allows the script to run without blocking in the notebook
result = subprocess.run(
    ['python', 'deploy_runtime.py', '--yes'],
    cwd='mcp-lakehouse-server',
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print('❌ Error:', result.stderr)
else:
    print('\n✅ MCP Server deployed!')
    print('\n📋 Runtime configuration saved to SSM Parameter Store')

MCP Athena Server Deployment to AgentCore Runtime

🔍 Loading configuration from SSM Parameter Store...
🔍 Using default AWS credentials (no profile specified)
⚠️  No AWS region configured, using default: us-east-1
   To set your region:
   - Environment variable: export AWS_DEFAULT_REGION=your-region
   - AWS CLI: aws configure set region your-region

✅ AWS Credentials Validated
   Region: us-east-1
   Account ID: XXXXXXXXXXXX
   Profile: default
   Auth method: AWS SSO

✅ Configuration loaded from SSM Parameter Store
   Region: us-east-1
   Account: XXXXXXXXXXXX
✅ Configuration validated

📋 Configuration Status:
   AWS Account: XXXXXXXXXXXX
   Region: us-east-1
   S3 Bucket: XXXXXXXXXXXX-us-east-1-agent
   Database: lakehouse_db
   RLS Role ARN: arn:aws:iam::ACCOUNT_ID:role/lakehouse-rls-role
   Cognito User Pool ARN: arn:aws:cognito-idp:us-east-1:XXXXXXXXXXXX:userpool/us-east-1_3jSGececJ
   Security Mode: lakeformation
   Log Level: DEBUG

✅ Auto-confirming deployment (--yes flag prov

## Step 2: Verify MCP Server Deployment

The deploy_runtime.py script automatically saves the Runtime ARN to SSM.
Run this cell to verify the deployment.

In [6]:
# Verify MCP Server Runtime configuration in SSM
print("Verifying MCP Server Runtime in SSM...\n")

parameters_to_check = [
    '/app/lakehouse-agent/mcp-server-runtime-arn',
    '/app/lakehouse-agent/mcp-server-runtime-id',
]

all_found = True
for param_name in parameters_to_check:
    try:
        response = ssm_client.get_parameter(Name=param_name)
        value = response['Parameter']['Value']
        print(f'✅ {param_name}')
        print(f'   Value: {value}')
    except ssm_client.exceptions.ParameterNotFound:
        print(f'❌ {param_name} - NOT FOUND')
        all_found = False
    except Exception as e:
        print(f'⚠️  {param_name} - ERROR: {e}')
        all_found = False

if all_found:
    print('\n✅ MCP Server Runtime configuration verified in SSM!')
else:
    print('\n⚠️  MCP Server Runtime parameters missing.')
    print('    The deploy_runtime.py script should have saved these automatically.')
    print('    Check the deployment output for errors.')

Verifying MCP Server Runtime in SSM...

✅ /app/lakehouse-agent/mcp-server-runtime-arn
   Value: arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:runtime/lakehouse_mcp_server-iPIYzlC3zh
✅ /app/lakehouse-agent/mcp-server-runtime-id
   Value: lakehouse_mcp_server-iPIYzlC3zh

✅ MCP Server Runtime configuration verified in SSM!


## Summary

✅ **MCP Server Deployment Complete!**

The MCP Server Runtime has been deployed and configuration saved to SSM Parameter Store.

**Next Steps:**
Run **04-deploy-gateway.ipynb** to deploy the AgentCore Gateway